# Clean Greater Sydney sales

Build the canonical 34-LGA lookup and a quarterly sales table from the original DCJ workbooks. Keep the published Greater Sydney and ring aggregates and use the first-published median for each quarter (D06). The table retains Wollondilly's missing 2019 unit values (D05).

The notebook reads the raw DCJ sales, ABS population table and ABS boundary export. Run from the repository root with the task's nbconvert command.

## 1. Read the DCJ workbooks

Read the `LGA` sheet and keep the columns needed for sales and revisions. Find the heading by its dwelling-type label because its Excel row varies. Read the quarter from the title in older files or the reporting-period line in newer files. Keep the original worksheet row number so exported values can be checked by hand.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display
from openpyxl import load_workbook

repo_root = Path("..") if Path.cwd().name == "notebooks" else Path(".")
raw_dir = repo_root / "data/raw"

raw_columns = [
    "gmr", "sydney_group", "ring", "lga", "source_dwelling",
    "first_quartile", "median_thousands", "third_quartile", "mean_thousands",
    "count_raw", "change_raw",
]

def read_sales_workbook(path):
    """Read one issue's reporting quarter and LGA table with Excel row numbers."""
    workbook = load_workbook(path, read_only=True, data_only=True)
    try:
        rows = list(workbook["LGA"].iter_rows(max_col=11, values_only=True))
    finally:
        workbook.close()

    caption = rows[0][0]
    if str(rows[1][0]).startswith("Reporting period:"):
        caption = rows[1][0]
    quarter_month = " ".join(caption.replace(" Quarter", "").split()[-2:])
    quarter_end_month = pd.to_datetime(quarter_month, format="%B %Y")
    quarter_date = quarter_end_month.to_period("Q").start_time

    header_index = next(index for index, row in enumerate(rows) if row[4] == "DwellingType")
    table = pd.DataFrame(rows[header_index + 1:], columns=raw_columns, dtype=object)
    first_data_row = header_index + 2
    table["source_row"] = range(first_data_row, first_data_row + len(table))
    table["source_file"] = path.name
    table["quarter_date"] = quarter_date
    table = table.loc[table["source_dwelling"].notna()]
    return table, header_index + 1

workbooks = sorted((raw_dir / "dcj_sales").glob("*.xlsx"))
tables = []
headers = []
for path in workbooks:
    table, header_row = read_sales_workbook(path)
    tables.append(table)
    headers.append(header_row)

raw_sales = pd.concat(tables, ignore_index=True)
print(f"{len(workbooks)} workbooks, {len(raw_sales):,} rows in their LGA tables.")
print("Workbooks by Excel header row")
display(pd.Series(headers, name="header_row").value_counts().sort_index().rename("workbooks"))

29 workbooks, 81,127 rows in their LGA tables.
Workbooks by Excel header row


header_row
5     2
6     1
7    26
Name: workbooks, dtype: int64

## 2. Select published Greater Sydney rows

Use column A = `GMR (Greater Metropolitan Region)` and B = `Greater Sydney`. Within that block, C identifies each ring. D gives the LGA name on an LGA row and `Total` on a ring aggregate row. The published Greater Sydney aggregate has `Total` in both C and D.

Keep column E's `Non Strata` and `Strata` labels as house and unit respectively. Use the source aggregate values directly as benchmarks.

In [2]:
rings = ["Inner Ring", "Middle Ring", "Outer Ring"]
dwelling_labels = {"Non Strata": "house", "Strata": "unit"}
sydney_block = raw_sales.loc[
    raw_sales["gmr"].eq("GMR (Greater Metropolitan Region)")
    & raw_sales["sydney_group"].eq("Greater Sydney")
]
named_ring = sydney_block["ring"].isin(rings)
greater_total = sydney_block["ring"].eq("Total") & sydney_block["lga"].eq("Total")
selected = sydney_block.loc[
    (named_ring | greater_total) & sydney_block["source_dwelling"].isin(dwelling_labels)
].copy()

selected["geo_level"] = "lga"
selected.loc[selected["lga"].eq("Total"), "geo_level"] = "ring"
selected.loc[selected["ring"].eq("Total"), "geo_level"] = "greater_sydney"
selected["geo_name"] = selected["lga"]
selected.loc[selected["geo_level"].eq("ring"), "geo_name"] = selected["ring"]
selected.loc[selected["geo_level"].eq("greater_sydney"), "geo_name"] = "Greater Sydney"
selected["region"] = selected["ring"].where(selected["geo_level"].ne("greater_sydney"))
selected["dwelling_type"] = selected["source_dwelling"].map(dwelling_labels)
print("Source rows retained before completing the grid")
display(selected.groupby("geo_level").size().rename("rows"))

Source rows retained before completing the grid


geo_level
greater_sydney      58
lga               1971
ring               174
Name: rows, dtype: int64

## 3. Convert values and record suppression

Column G is the median in thousands of dollars and J is the number of sales. Strip commas from numeric text, and represent a dash or small-sample marker as missing in the numeric fields. Keep the original marker meaning in a status field. Parse K for the revision check. Excel numeric changes are already fractions, and text percentages are divided by 100.

In [3]:
def parse_number(values):
    """Read an Excel measure with numeric text, commas or DCJ markers."""
    text = values.astype("string").str.strip().str.replace(",", "", regex=False)
    return pd.to_numeric(text.mask(text.isin(["-", "s"])), errors="raise")

selected["median_price"] = (parse_number(selected["median_thousands"]) * 1000).astype("Int64")
selected["sales_count"] = parse_number(selected["count_raw"]).astype("Int64")
selected["price_status"] = "published"
selected.loc[selected["median_thousands"].eq("-"), "price_status"] = "suppressed"
selected["count_status"] = "published"
selected.loc[selected["count_raw"].eq("-"), "count_status"] = "suppressed_10_or_fewer"
selected.loc[selected["count_raw"].eq("s"), "count_status"] = "suppressed_30_or_fewer"

assert selected["median_price"].isna().eq(selected["price_status"].ne("published")).all()
assert selected["sales_count"].isna().eq(selected["count_status"].ne("published")).all()

change_text = selected["change_raw"].astype("string").str.strip()
percent_text = change_text.str.endswith("%", na=False)
quarterly_change = parse_number(change_text.str.removesuffix("%"))
quarterly_change.loc[percent_text] = quarterly_change.loc[percent_text] / 100
selected["quarterly_change"] = quarterly_change
print("Source numeric values retained")
display(selected[["median_price", "sales_count", "quarterly_change"]].count())

Source numeric values retained


median_price        2188
sales_count         2058
quarterly_change    2188
dtype: int64

## 4. Complete the quarterly grid

Make one expected row for every combination of workbook quarter, geography and house or unit. Attach the published values and label an unmatched row `not_in_source`. Keep its workbook name for traceability and leave its Excel row blank. Flag a row when DCJ withholds the sales count or the row is absent.

In [4]:
keys = ["quarter_date", "geo_level", "geo_name", "dwelling_type"]
quarters = raw_sales[["quarter_date", "source_file"]].drop_duplicates()
geographies = selected[["geo_level", "geo_name", "region"]].drop_duplicates()
dwellings = pd.DataFrame({"dwelling_type": list(dwelling_labels.values())})

grid = quarters.merge(geographies, how="cross").merge(dwellings, how="cross")
source_values = keys + [
    "median_price", "sales_count", "price_status", "count_status", "source_row", "quarterly_change",
]
sales = grid.merge(selected[source_values], on=keys, how="left", validate="one_to_one", indicator=True)
not_in_source = sales.pop("_merge").eq("left_only")
sales.loc[not_in_source, ["price_status", "count_status"]] = "not_in_source"
sales["source_row"] = sales["source_row"].astype("Int64")
sales["low_sales_flag"] = sales["count_status"].ne("published")
sales = sales.sort_values(keys, ignore_index=True)

assert sales["source_row"].notna().sum() == len(selected)
print("Rows added for combinations absent from the workbooks")
display(sales.loc[sales["price_status"].eq("not_in_source"), [
    "quarter_date", "geo_name", "dwelling_type", "source_file",
]])

Rows added for combinations absent from the workbooks


,quarter_date,geo_name,dwelling_type,source_file
143,2019-04-01,Wollondilly,unit,issue-129-sales-tables-june-2019.xlsx


## 5. Build the LGA lookup

Use the canonical DCJ label and ring from the raw LGA rows. Check every quarter's ring assignment. Match the six spelling differences explicitly to the ABS labels in ERP Table 1, then retain the five-digit LGA 2025 code for later joins. Check that each code also occurs in the T02 boundary export.

In [5]:
dcj_lgas = selected.loc[selected["geo_level"].eq("lga"), ["quarter_date", "geo_name", "region"]]
membership = dcj_lgas.drop_duplicates().rename(columns={"geo_name": "lga"})
per_lga = membership.groupby("lga").agg(quarters=("quarter_date", "nunique"), rings=("region", "nunique"))
assert len(per_lga) == 34
assert per_lga["quarters"].eq(29).all() and per_lga["rings"].eq(1).all()

dcj_to_abs_name = {
    "Bayside": "Bayside (NSW)",
    "Campbelltown": "Campbelltown (NSW)",
    "Central Coast": "Central Coast (NSW)",
    "Ku-Ring-Gai": "Ku-ring-gai",
    "Sutherland Shire": "Sutherland",
    "The Hills Shire": "The Hills",
}
lga_lookup = membership[["lga", "region"]].drop_duplicates()
lga_lookup["abs_name_2025"] = lga_lookup["lga"].replace(dcj_to_abs_name)
assert dcj_to_abs_name.keys() <= set(lga_lookup["lga"])

erp = pd.read_excel(
    raw_dir / "abs_erp/32180DS0004_2001-25.xlsx", sheet_name="Table 1",
    header=None, skiprows=6, usecols="A:B",
    names=["lga_code_2025", "abs_name_2025"], dtype="string",
)
erp = erp.loc[erp["lga_code_2025"].str.fullmatch(r"\d{5}", na=False)]
lga_lookup = lga_lookup.merge(erp, on="abs_name_2025", how="left", validate="one_to_one")
lookup_columns = ["lga", "region", "lga_code_2025", "abs_name_2025"]
lga_lookup = lga_lookup[lookup_columns].sort_values("lga", ignore_index=True)
codes = set(lga_lookup["lga_code_2025"])
assert len(lga_lookup) == len(codes) == 34
assert lga_lookup["lga_code_2025"].str.fullmatch(r"\d{5}", na=False).all()
assert codes <= set(erp["lga_code_2025"])

with (raw_dir / "abs_asgs/LGA_2025_Sydney_region.geojson").open(encoding="utf-8") as stream:
    features = json.load(stream)["features"]
boundary_codes = {feature["properties"]["lga_code_2025"] for feature in features}
assert codes <= boundary_codes
ring_counts = lga_lookup["region"].value_counts().reindex(rings)
assert ring_counts.to_dict() == {"Inner Ring": 8, "Middle Ring": 12, "Outer Ring": 14}
print("No LGA changed rings across the 29 quarters. All 34 codes occur in ERP and GeoJSON.")
display(ring_counts.rename("lgas"))
display(lga_lookup.loc[lga_lookup["lga"].isin(dcj_to_abs_name)].reset_index(drop=True))

No LGA changed rings across the 29 quarters. All 34 codes occur in ERP and GeoJSON.


region
Inner Ring      8
Middle Ring    12
Outer Ring     14
Name: lgas, dtype: int64

,lga,region,lga_code_2025,abs_name_2025
0,Bayside,Middle Ring,10500,Bayside (NSW)
1,Campbelltown,Outer Ring,11500,Campbelltown (NSW)
2,Central Coast,Outer Ring,11650,Central Coast (NSW)
3,Ku-Ring-Gai,Middle Ring,14500,Ku-ring-gai
4,Sutherland Shire,Outer Ring,17150,Sutherland
5,The Hills Shire,Outer Ring,17420,The Hills


## 6. Measure revisions (D06)

Compare each later issue with the first-published median for the immediately preceding quarter. Sort the geography and dwelling-type series by quarter before shifting one row. The completed grid preserves that alignment when a source row is missing. Count only pairs with both medians and a usable quarterly change. Analyse LGAs and Greater Sydney separately.

Let `A` be the current published median in dollars, `B` the first-published previous median in dollars and `q = 1 + change` the published quarterly ratio. The implied revised prior median is `A / q`. The point estimate of the revision is `A / (q × B) − 1`.

All selected medians are published in whole thousands of dollars, and all published changes can be expressed to 0.01 percentage point. Under nearest rounding, allow each median ±$500 and allow `q` ±0.00005. If the underlying prior median were unchanged, it would lie in `[B − 500, B + 500]`. The smallest and largest compatible displayed current medians would then be `(B − 500) × (q − 0.00005) − 500` and `(B + 500) × (q + 0.00005) + 500`. Divide both endpoints by `q × B` and subtract one to get the rounding-only revision bounds for that pair. This uses the precision of both prices and the change. Smaller medians have a wider percentage bound.

In [6]:
comparison_levels = ["lga", "greater_sydney"]
comparisons = sales.loc[sales["geo_level"].isin(comparison_levels)].copy()
series_keys = ["geo_level", "geo_name", "dwelling_type"]
previous_columns = ["quarter_date", "median_price", "sales_count", "source_file", "source_row"]
series = comparisons.sort_values("quarter_date").groupby(series_keys)
previous = series[previous_columns].shift(1).add_prefix("previous_")
comparisons = comparisons.join(previous)
comparisons = comparisons.loc[comparisons["previous_quarter_date"].notna()].copy()
quarter_numbers = comparisons["quarter_date"].dt.to_period("Q").astype("int64")
previous_numbers = comparisons["previous_quarter_date"].dt.to_period("Q").astype("int64")
assert quarter_numbers.sub(previous_numbers).eq(1).all()

known_changes = selected["quarterly_change"].dropna()
assert (known_changes * 10_000).sub((known_changes * 10_000).round()).abs().lt(1e-7).all()
assert selected["median_price"].dropna().mod(1000).eq(0).all()

price_pairs = comparisons.dropna(subset=["median_price", "previous_median_price"])
revisions = price_pairs.dropna(subset=["quarterly_change"]).copy()
price_rounding = 500
change_rounding = 0.00005

ratio = 1 + revisions["quarterly_change"]
assert ratio.gt(change_rounding).all()
prior_price = revisions["previous_median_price"]
revisions["implied_revised_prev"] = revisions["median_price"] / ratio
revisions["revision_pct"] = (revisions["implied_revised_prev"] / prior_price - 1) * 100
revisions["absolute_revision_pct"] = revisions["revision_pct"].abs()

minimum_display = (prior_price - price_rounding) * (ratio - change_rounding) - price_rounding
maximum_display = (prior_price + price_rounding) * (ratio + change_rounding) + price_rounding
lower_bound_pct = (minimum_display / (ratio * prior_price) - 1) * 100
upper_bound_pct = (maximum_display / (ratio * prior_price) - 1) * 100
revisions["within_rounding"] = revisions["revision_pct"].between(lower_bound_pct, upper_bound_pct)
revisions["max_rounding_pct"] = pd.concat([lower_bound_pct.abs(), upper_bound_pct.abs()], axis=1).max(axis=1)

summary_keys = ["geo_level", "dwelling_type"]
revision_summary = revisions.groupby(summary_keys).agg(
    compared=("revision_pct", "size"),
    median_abs_pct=("absolute_revision_pct", "median"),
    p90_abs_pct=("absolute_revision_pct", lambda values: values.quantile(0.90)),
    max_abs_pct=("absolute_revision_pct", "max"),
    within_rounding_pct=("within_rounding", "mean"),
    max_rounding_pct=("max_rounding_pct", "max"),
)
revision_summary["within_rounding_pct"] *= 100
revision_summary.insert(0, "possible", comparisons.groupby(summary_keys).size())
print("Revision and rounding figures are in percent.")
display(revision_summary.round(4).T)

Revision and rounding figures are in percent.


geo_level           greater_sydney               lga         
dwelling_type                house     unit    house     unit
possible                        28       28      952      952
compared                        28       28      952      932
median_abs_pct               1.848   0.4475   0.5747   0.2868
p90_abs_pct                 2.8272   0.8399    3.976   2.3344
max_abs_pct                 3.5991   1.2821  39.1635   18.741
within_rounding_pct         7.1429  28.5714  26.9958  42.8112
max_rounding_pct            0.1199   0.1487   0.1805   0.2487

### Largest revisions and relevance to the project windows

Show the five largest absolute revisions for LGAs and for Greater Sydney. Keep the sign, the implied median and the worksheet references. Check which revisions affect quarters included in the 2019 baseline or in the latest four-quarter window. This comparison uses revisions within those windows and leaves the growth rankings for the later analysis task.

In [7]:
sorted_revisions = revisions.sort_values("absolute_revision_pct", ascending=False, kind="stable")
for level in comparison_levels:
    largest = sorted_revisions.loc[sorted_revisions["geo_level"].eq(level)].head(5).copy()
    largest.index = range(1, len(largest) + 1)
    print(level, "five largest absolute revisions")
    display(largest[[
        "geo_name", "dwelling_type", "previous_quarter_date", "previous_median_price",
        "implied_revised_prev", "revision_pct",
    ]].round({"implied_revised_prev": 0, "revision_pct": 3}))

    prior_cell = "G" + largest["previous_source_row"].astype("string")
    current_row = largest["source_row"].astype("string")
    display(pd.DataFrame({
        "previous_issue": largest["previous_source_file"],
        "previous_cell": "LGA!" + prior_cell,
        "current_issue": largest["source_file"],
        "current_cells": "LGA!G" + current_row + ", K" + current_row,
    }))

latest_four_quarters = set(quarters["quarter_date"].sort_values().tail(4))
lga_revisions = revisions.loc[revisions["geo_level"].eq("lga")]
for period, window in [
    ("2019 baseline", lga_revisions["previous_quarter_date"].dt.year.eq(2019)),
    ("Latest four quarters", lga_revisions["previous_quarter_date"].isin(latest_four_quarters)),
]:
    in_window = lga_revisions.loc[window]
    print(period, "revisable LGA quarters through", in_window["previous_quarter_date"].max().date())
    display(in_window.groupby("dwelling_type")["absolute_revision_pct"].agg(
        pairs="size", median_abs_pct="median", max_abs_pct="max",
    ).round(3))

lga five largest absolute revisions


,geo_name,dwelling_type,previous_quarter_date,previous_median_price,implied_revised_prev,revision_pct
1,Woollahra,house,2025-04-01,4958000,6899724.0,39.163
2,Woollahra,house,2022-10-01,4175000,5300036.0,26.947
3,Woollahra,house,2023-04-01,4240000,5349929.0,26.178
4,Woollahra,house,2021-10-01,4250000,5299685.0,24.698
5,Mosman,house,2023-04-01,4300000,5312532.0,23.547


,previous_issue,previous_cell,current_issue,current_cells
1,sales_tables_june_2025_quarter.xlsx,LGA!G1796,sales_tables_september_2025_quarter.xlsx,"LGA!G1824, K1824"
2,issue-143-sales-tables-december-2022.xlsx,LGA!G1796,issue-144-sales-tables-march-2023.xlsx,"LGA!G1800, K1800"
3,issue-145-sales-tables-june-2023.xlsx,LGA!G1808,issue-146-sales-tables-september-2023.xlsx,"LGA!G1808, K1808"
4,issue-139-sales-tables-december-2021.xlsx,LGA!G1828,issue-140-sales-tables-march-2022.xlsx,"LGA!G1808, K1808"
5,issue-145-sales-tables-june-2023.xlsx,LGA!G1793,issue-146-sales-tables-september-2023.xlsx,"LGA!G1793, K1793"


greater_sydney five largest absolute revisions


,geo_name,dwelling_type,previous_quarter_date,previous_median_price,implied_revised_prev,revision_pct
1,Greater Sydney,house,2021-01-01,1110000,1149950.0,3.599
2,Greater Sydney,house,2024-10-01,1452000,1500052.0,3.309
3,Greater Sydney,house,2019-01-01,865000,890032.0,2.894
4,Greater Sydney,house,2025-01-01,1430000,1470021.0,2.799
5,Greater Sydney,house,2024-04-01,1460000,1500265.0,2.758


,previous_issue,previous_cell,current_issue,current_cells
1,issue-136-sales-tables-march-2021.xlsx,LGA!G1695,issue-137-sales-tables-june-2021.xlsx,"LGA!G1695, K1695"
2,issue-151-sales-tables-dec-2024.xlsx,LGA!G1679,issue-152-sales-tables-mar-2025.xlsx,"LGA!G1671, K1671"
3,issue-128-sales-tables-march-2019.xlsx,LGA!G1649,issue-129-sales-tables-june-2019.xlsx,"LGA!G1647, K1647"
4,issue-152-sales-tables-mar-2025.xlsx,LGA!G1671,sales_tables_june_2025_quarter.xlsx,"LGA!G1667, K1667"
5,issue-149-sales-tables-june-2024-quarter.xlsx,LGA!G1675,Sales_tables_September_2024_quarter.xlsx,"LGA!G1675, K1675"


2019 baseline revisable LGA quarters through 2019-10-01


,pairs,median_abs_pct,max_abs_pct
dwelling_type,,,
house,136,0.613,13.287
unit,132,0.467,8.925


Latest four quarters revisable LGA quarters through 2025-10-01


,pairs,median_abs_pct,max_abs_pct
dwelling_type,,,
house,102,0.717,39.163
unit,102,0.285,11.001


### Limitations wording

DCJ can revise quarterly medians after release, so this project consistently uses first-published values. The median absolute implied revision was 0.57% for LGA houses and 0.29% for units, with a 39.16% house outlier in the latest four-quarter window. These revisions could move close rankings and affect LGAs with unusually large revised quarters.

## 7. Validate the sales table

Check the complete grid, quarter coverage, geography lookup, missing-value statuses and plausible price range. Compare six values or statuses with the cells reviewed in T01. Then reconcile with the marker counts from `01_explore.ipynb` section 6, which included the all-dwelling `Total` rows and, for its full-sheet count, repeated geography levels.

In [8]:
expected_counts = {"greater_sydney": 58, "lga": 1972, "ring": 174}
assert sales.groupby("geo_level").size().to_dict() == expected_counts
assert not sales.duplicated(keys).any()

expected_quarters = pd.date_range("2019-01-01", "2026-01-01", freq="QS")
actual_quarters = pd.DatetimeIndex(quarters["quarter_date"].sort_values())
assert actual_quarters.equals(expected_quarters)
lga_sales = sales.loc[sales["geo_level"].eq("lga")]
assert set(lga_sales["geo_name"]) == set(lga_lookup["lga"])

absent = sales.loc[sales["price_status"].eq("not_in_source")]
assert len(absent) == 1
assert absent[keys].iloc[0].tolist() == [pd.Timestamp("2019-04-01"), "lga", "Wollondilly", "unit"]
assert sales["source_row"].isna().eq(sales["price_status"].eq("not_in_source")).all()
assert sales["source_row"].isna().eq(sales["count_status"].eq("not_in_source")).all()
assert sales["source_file"].notna().all()
assert sales["median_price"].isna().eq(sales["price_status"].ne("published")).all()
assert sales["sales_count"].isna().eq(sales["count_status"].ne("published")).all()
assert sales["low_sales_flag"].eq(sales["count_status"].ne("published")).all()
assert sales["median_price"].dropna().between(100_000, 10_000_000).all()
assert sales["sales_count"].dropna().gt(0).all()
assert sales["region"].isna().eq(sales["geo_level"].eq("greater_sydney")).all()

checks = [
    ("2026-01-01", "lga", "Blacktown", "house", "median_price", 1_250_000),
    ("2026-01-01", "lga", "Sydney", "unit", "median_price", 967_000),
    ("2026-01-01", "lga", "Parramatta", "unit", "sales_count", 835),
    ("2026-01-01", "greater_sydney", "Greater Sydney", "house", "median_price", 1_550_000),
    ("2026-01-01", "greater_sydney", "Greater Sydney", "unit", "median_price", 855_000),
    ("2019-01-01", "lga", "Wollondilly", "unit", "price_status", "suppressed"),
]
indexed_sales = sales.set_index(keys)
for date, level, name, dwelling, field, expected in checks:
    result = indexed_sales.loc[(pd.Timestamp(date), level, name, dwelling)]
    assert result[field] == expected, (date, name, dwelling, field, result[field])
    column = "J" if field == "sales_count" else "G"
    print(f"{date} {name} {dwelling}: {expected} ({result['source_file']}, LGA!{column}{result['source_row']})")

print("Rows by geography", expected_counts)
print("All key, date, missing-value, price-range and spot checks passed.")
print("LGA price statuses")
display(lga_sales.groupby(["dwelling_type", "price_status"]).size().unstack(fill_value=0))
print("LGA count statuses")
display(lga_sales.groupby(["dwelling_type", "count_status"]).size().unstack(fill_value=0))
print("Published price minimum and maximum in dollars",
      sales["median_price"].min(), sales["median_price"].max())

2026-01-01 Blacktown house: 1250000 (sales-tables-march-2026-quarter.xlsx, LGA!G1865)
2026-01-01 Sydney unit: 967000 (sales-tables-march-2026-quarter.xlsx, LGA!G1815)
2026-01-01 Parramatta unit: 835 (sales-tables-march-2026-quarter.xlsx, LGA!J1851)
2026-01-01 Greater Sydney house: 1550000 (sales-tables-march-2026-quarter.xlsx, LGA!G1691)
2026-01-01 Greater Sydney unit: 855000 (sales-tables-march-2026-quarter.xlsx, LGA!G1692)
2019-01-01 Wollondilly unit: suppressed (issue-128-sales-tables-march-2019.xlsx, LGA!G1863)
Rows by geography {'greater_sydney': 58, 'lga': 1972, 'ring': 174}
All key, date, missing-value, price-range and spot checks passed.
LGA price statuses


price_status,not_in_source,published,suppressed
dwelling_type,,,
house,0,986,0
unit,1,970,15


LGA count statuses


count_status,not_in_source,published,suppressed_10_or_fewer,suppressed_30_or_fewer
dwelling_type,,,,
house,0,949,0,37
unit,1,877,15,93


Published price minimum and maximum in dollars 404000 6750000


### Reconcile with the raw marker counts

Recount the source markers at the same scope as the T01 audit. Compare the Sydney LGA counts before and after removing the `Total` dwelling rows. Published ring and Greater Sydney rows are part of the export, so check those for additional markers. The explicitly added absent row carries no raw marker.

In [9]:
def suppression_counts(table):
    """Count the original median and sales-count markers without changing values."""
    return {
        "price_dash": int(table["median_thousands"].eq("-").sum()),
        "count_dash": int(table["count_raw"].eq("-").sum()),
        "count_s": int(table["count_raw"].eq("s").sum()),
    }

all_sheet = suppression_counts(raw_sales)
sydney_all_types = sydney_block.loc[sydney_block["ring"].isin(rings) & sydney_block["lga"].ne("Total")]
sydney_lga_markers = suppression_counts(sydney_all_types)
sydney_total_markers = suppression_counts(sydney_all_types.loc[sydney_all_types["source_dwelling"].eq("Total")])
retained_markers = suppression_counts(selected.loc[selected["geo_level"].eq("lga")])
aggregate_markers = suppression_counts(selected.loc[selected["geo_level"].ne("lga")])
export_markers = {
    "price_dash": int(sales["price_status"].eq("suppressed").sum()),
    "count_dash": int(sales["count_status"].eq("suppressed_10_or_fewer").sum()),
    "count_s": int(sales["count_status"].eq("suppressed_30_or_fewer").sum()),
}
assert all_sheet == {"price_dash": 8016, "count_dash": 8016, "count_s": 12992}
assert sydney_lga_markers == {"price_dash": 15, "count_dash": 15, "count_s": 133}
assert sydney_total_markers == {"price_dash": 0, "count_dash": 0, "count_s": 3}
assert aggregate_markers == {"price_dash": 0, "count_dash": 0, "count_s": 0}
assert retained_markers == export_markers == {"price_dash": 15, "count_dash": 15, "count_s": 130}
display(pd.DataFrame.from_dict({
    "T01 whole LGA sheet": all_sheet,
    "T01 Sydney LGA rows": sydney_lga_markers,
    "Removed Sydney LGA Total rows": sydney_total_markers,
    "Retained ring and Sydney aggregate rows": aggregate_markers,
    "Clean export": export_markers,
}, orient="index"))

,price_dash,count_dash,count_s
T01 whole LGA sheet,8016,8016,12992
T01 Sydney LGA rows,15,15,133
Removed Sydney LGA Total rows,0,0,3
Retained ring and Sydney aggregate rows,0,0,0
Clean export,15,15,130


### Check large adjacent-quarter moves

Look for absolute first-published median moves above 25% when the current LGA-quarter has more than 30 published sales. Also count how many have more than 30 recorded sales in the prior quarter. These examples are for source review. They leave the published values in the export.

In [10]:
lga_price_pairs = price_pairs.loc[price_pairs["geo_level"].eq("lga")].copy()
lga_price_pairs["move_pct"] = (
    lga_price_pairs["median_price"] / lga_price_pairs["previous_median_price"] - 1
) * 100
lga_price_pairs["abs_move_pct"] = lga_price_pairs["move_pct"].abs()
large_moves = lga_price_pairs.loc[
    lga_price_pairs["abs_move_pct"].gt(25) & lga_price_pairs["sales_count"].gt(30)
].copy()
both_counts = large_moves.loc[large_moves["previous_sales_count"].gt(30)]
print("Moves above 25% with more than 30 sales")
display(pd.DataFrame({
    "current_quarter": large_moves.groupby("dwelling_type").size(),
    "both_quarters": both_counts.groupby("dwelling_type").size(),
}).reindex(dwellings["dwelling_type"], fill_value=0).fillna(0).astype(int))
print("Five largest moves with more than 30 recorded sales in both quarters")
display(both_counts.nlargest(5, "abs_move_pct")[[
    "quarter_date", "geo_name", "dwelling_type", "previous_median_price",
    "median_price", "previous_sales_count", "sales_count", "move_pct",
]].round({"move_pct": 2}))

Moves above 25% with more than 30 sales


,current_quarter,both_quarters
dwelling_type,,
house,8,7
unit,5,4


Five largest moves with more than 30 recorded sales in both quarters


,quarter_date,geo_name,dwelling_type,previous_median_price,median_price,previous_sales_count,sales_count,move_pct
1053,2022-04-01,Willoughby,unit,1100000,1620000,128,205,47.27
1588,2024-01-01,Woollahra,house,6750000,4100000,105,95,-39.26
1512,2023-10-01,Woollahra,house,4900000,6750000,108,105,37.76
1176,2022-10-01,Lane Cove,house,4575000,2875000,78,57,-37.16
1029,2022-04-01,Mosman,unit,1200000,1600000,80,95,33.33


## 8. Export the two CSVs

Write the validated lookup and sales table in a deterministic order and keep missing cells empty. Export dates as ISO dates and keep the 2025 codes as five characters. The revision estimates and anomaly checks stay in the notebook.

In [11]:
sales_columns = [
    "quarter_date", "geo_level", "geo_name", "region", "dwelling_type",
    "median_price", "sales_count", "price_status", "count_status", "low_sales_flag",
    "source_file", "source_row",
]
export_paths = {
    repo_root / "data/reference/lga_lookup.csv": lga_lookup,
    repo_root / "data/interim/sales_long.csv": sales[sales_columns],
}
for path, table in export_paths.items():
    path.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(path, index=False, encoding="utf-8", date_format="%Y-%m-%d", lineterminator="\n")
    print(f"Wrote {path.relative_to(repo_root)} with {len(table):,} rows.")

Wrote data/reference/lga_lookup.csv with 34 rows.
Wrote data/interim/sales_long.csv with 2,204 rows.


### Output columns

`lga_lookup.csv`

- `lga` — canonical DCJ local government area name.
- `region` — its DCJ Inner, Middle or Outer Ring.
- `lga_code_2025` — five-character ABS LGA 2025 identifier.
- `abs_name_2025` — ABS label for that code.

`sales_long.csv`

- `quarter_date` — first day of the reference quarter in YYYY-MM-DD format.
- `geo_level` — `lga`, `ring` or `greater_sydney`.
- `geo_name` — canonical LGA, ring or Greater Sydney name.
- `region` — DCJ ring, blank for Greater Sydney.
- `dwelling_type` — `house` for non-strata or `unit` for strata.
- `median_price` — first-published median in Australian dollars, blank when unavailable.
- `sales_count` — published sales count, blank when unavailable.
- `price_status` — `published`, `suppressed` or `not_in_source`.
- `count_status` — `published`, `suppressed_10_or_fewer`, `suppressed_30_or_fewer` or `not_in_source`.
- `low_sales_flag` — true if the count is withheld or the row is absent.
- `source_file` — name of the DCJ workbook for the quarter.
- `source_row` — Excel row number in `LGA`, blank when the row is absent.